# Random rollout on CartPole

This notebook shows the smallest end-to-end loop: build a **vector** environment from `EnvConfig`, sample random actions, and inspect the plain dict records returned each step.

See [docs/guide.md](../docs/guide.md) for the full step contract.

## Imports

`EnvConfig` accepts any Gymnasium environment id through `id`. `make_vector_env` wraps Gymnasium and formats steps as `(result, metrics)`.

In [1]:
from mouse_envs import EnvConfig, make_vector_env

## Configure the environment

- `num_envs=1` runs one CartPole instance.
- `max_episode_steps` caps episode length (truncation).
- `seed` fixes RNG for reproducibility.

In [2]:
cfg = EnvConfig(
    id="CartPole-v1",
    seed=0,
    num_envs=1,
    max_episode_steps=500,
)
env = make_vector_env(cfg)
env.name

'CartPole-v1#0'

## Rollout loop

Environment names are available by vector index as `env.names`; `env.name` returns the first one for single-env use.

Each `env.step` returns:

- **`result[i]`** — `time`, `observation` (dict: `discrete` / `continuous` / `image`), `reward`, `done`, `episode_index`, `reward_episodic`, optional `q_star` / `ns_params`
- **`metrics[i]`** — `episode_cum_reward` and `episode_length` lists for finishes on this step (empty if none)

Each **`actions[i]["action"]`** passed to `step()` is also a dict — `discrete` or `continuous`, matching the env's action space.

The **first** `step` call performs an internal reset (actions are ignored); see the guide for the initial frame (`time == 0`).

In [3]:
for step in range(1000):
    actions = env.sample_random_actions()
    result, metrics = env.step(actions)
    print(actions[0])
    print(result[0])

{'action': {'discrete': tensor(1)}}
{'time': tensor(0), 'observation': {'continuous': tensor([-0.0147,  0.0180,  0.0376, -0.0261])}, 'reward': tensor(0.), 'done': tensor(0), 'episode_index': 0, 'reward_episodic': 0.0}
{'action': {'discrete': tensor(0)}}
{'time': tensor(1), 'observation': {'continuous': tensor([-0.0143, -0.1776,  0.0370,  0.2781])}, 'reward': tensor(1.), 'done': tensor(0), 'episode_index': 0, 'reward_episodic': 0.002}
{'action': {'discrete': tensor(1)}}
{'time': tensor(2), 'observation': {'continuous': tensor([-0.0179,  0.0169,  0.0426, -0.0026])}, 'reward': tensor(1.), 'done': tensor(0), 'episode_index': 0, 'reward_episodic': 0.006}
{'action': {'discrete': tensor(1)}}
{'time': tensor(3), 'observation': {'continuous': tensor([-0.0175,  0.2114,  0.0426, -0.2816])}, 'reward': tensor(1.), 'done': tensor(0), 'episode_index': 0, 'reward_episodic': 0.01}
{'action': {'discrete': tensor(0)}}
{'time': tensor(4), 'observation': {'continuous': tensor([-0.0133,  0.0157,  0.0369,  0

## Cleanup

In [4]:
env.close()